In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
data = pd.read_csv(
    "../data/raw/File.csv",
    encoding="cp1251",
    sep=";",
    on_bad_lines="skip",
    low_memory=False
)

In [3]:
print(data.head(10))
print(data.shape)

   НомерСтроки               Период  \
0            1   01.01.2024 0:00:00   
1            2   01.01.2024 0:00:00   
2            3   01.01.2024 0:00:00   
3            1  01.01.2024 12:00:00   
4            1   03.01.2024 9:58:36   
5            2   03.01.2024 9:58:36   
6            1  03.01.2024 10:01:11   
7            1  03.01.2024 10:38:01   
8            2  03.01.2024 10:38:01   
9            1  03.01.2024 10:42:24   

                                         Регистратор  \
0  Операция по единому налоговому счету 00БП-0000...   
1  Операция по единому налоговому счету 00БП-0000...   
2  Операция по единому налоговому счету 00БП-0000...   
3  Списание с расчетного счета 00БП-000001 от 01....   
4  Реализация (акт, накладная, УПД) 00УТ-000011 о...   
5  Реализация (акт, накладная, УПД) 00УТ-000011 о...   
6  Поступление наличных 00УТ-000001 от 03.01.2024...   
7  Реализация (акт, накладная, УПД) 00УТ-000012 о...   
8  Реализация (акт, накладная, УПД) 00УТ-000012 о...   
9  Реализа

In [4]:
print(data.columns.to_list)

<bound method IndexOpsMixin.tolist of Index(['НомерСтроки', 'Период', 'Регистратор', 'Контрагент', 'КонтрагентИНН',
       'СчетДт', 'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2',
       'ВидСубконтоДт2', 'СубконтоДт3', 'ВидСубконтоДт3', 'СчетКт',
       'СубконтоКт1', 'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2',
       'СубконтоКт3', 'ВидСубконтоКт3', 'Организация', 'ВалютаДт', 'ВалютаКт',
       'ПодразделениеДт', 'ПодразделениеКт', 'Сумма', 'ВалютнаяСуммаДт',
       'ВалютнаяСуммаКт', 'КоличествоДт', 'КоличествоКт', 'СуммаНУДт',
       'СуммаНУКт', 'СуммаПРДт', 'СуммаПРКт', 'СуммаВРДт', 'СуммаВРКт',
       'Содержание', 'НеКорректироватьСтоимостьАвтоматически', 'НадписьНУ',
       'НадписьПР', 'НадписьВР', 'НадписьКоличествоДт', 'НадписьКоличествоКт',
       'СчетДтКоличественный', 'СчетКтКоличественный', 'СчетДтВалютный',
       'СчетКтВалютный', 'СчетДтУчетПоПодразделениям',
       'СчетКтУчетПоПодразделениям', 'Unnamed: 48'],
      dtype='object')>


In [5]:
data['Период'] = pd.to_datetime(data['Период'], dayfirst=True, errors='coerce')


data["ВалютнаяСуммаДт_original"] = data["ВалютнаяСуммаДт"].copy()


data["Сумма"] = (
    data["ВалютнаяСуммаДт"]
    .astype(str)
    .str.replace(r'[\s\xa0\u202f\u2009]', '', regex=True)  # все виды пробелов
    .str.replace(",", ".", regex=False)
    .str.strip()
)
data["Сумма"] = pd.to_numeric(data["Сумма"], errors="coerce")


data["ТипДокумента"] = data["Регистратор"].str.extract(r"^(.+?)\s+\d{2}")
data["ТипДокумента"] = data["ТипДокумента"].str.strip()


manual_types = ['Операция']
data['is_manual'] = data['ТипДокумента'].isin(manual_types).astype(int)

print(data.columns.to_list)

<bound method IndexOpsMixin.tolist of Index(['НомерСтроки', 'Период', 'Регистратор', 'Контрагент', 'КонтрагентИНН',
       'СчетДт', 'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2',
       'ВидСубконтоДт2', 'СубконтоДт3', 'ВидСубконтоДт3', 'СчетКт',
       'СубконтоКт1', 'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2',
       'СубконтоКт3', 'ВидСубконтоКт3', 'Организация', 'ВалютаДт', 'ВалютаКт',
       'ПодразделениеДт', 'ПодразделениеКт', 'Сумма', 'ВалютнаяСуммаДт',
       'ВалютнаяСуммаКт', 'КоличествоДт', 'КоличествоКт', 'СуммаНУДт',
       'СуммаНУКт', 'СуммаПРДт', 'СуммаПРКт', 'СуммаВРДт', 'СуммаВРКт',
       'Содержание', 'НеКорректироватьСтоимостьАвтоматически', 'НадписьНУ',
       'НадписьПР', 'НадписьВР', 'НадписьКоличествоДт', 'НадписьКоличествоКт',
       'СчетДтКоличественный', 'СчетКтКоличественный', 'СчетДтВалютный',
       'СчетКтВалютный', 'СчетДтУчетПоПодразделениям',
       'СчетКтУчетПоПодразделениям', 'Unnamed: 48', 'ВалютнаяСуммаДт_original',
       'ТипДокумента

# Удаление

In [6]:
cols_to_drop = [
    "НомерСтроки",
    "Unnamed: 48",
    "НеКорректироватьСтоимостьАвтоматически",
    "НадписьНУ", "НадписьПР", "НадписьВР",
    "НадписьКоличествоДт", "НадписьКоличествоКт",
    "СчетДтКоличественный", "СчетКтКоличественный",
    "СчетДтВалютный", "СчетКтВалютный",
    "СчетДтУчетПоПодразделениям", "СчетКтУчетПоПодразделениям",
    "ВалютаДт", "ВалютаКт", "ВалютнаяСуммаКт",
    "СуммаНУДт", "СуммаНУКт",
    "СуммаПРДт", "СуммаПРКт",
    "СуммаВРДт", "СуммаВРКт",
    "Организация",
    "ВалютнаяСуммаДт",          # уже перенесли в Сумма
    "ВалютнаяСуммаДт_original", # служебная, больше не нужна
]

data = data.drop(columns=cols_to_drop, errors="ignore")
print(f"Осталось колонок: {data.shape[1]}")
print(data.columns.tolist())

Осталось колонок: 26
['Период', 'Регистратор', 'Контрагент', 'КонтрагентИНН', 'СчетДт', 'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2', 'ВидСубконтоДт2', 'СубконтоДт3', 'ВидСубконтоДт3', 'СчетКт', 'СубконтоКт1', 'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2', 'СубконтоКт3', 'ВидСубконтоКт3', 'ПодразделениеДт', 'ПодразделениеКт', 'Сумма', 'КоличествоДт', 'КоличествоКт', 'Содержание', 'ТипДокумента', 'is_manual']


# Заполнене пропусков и удаление

In [7]:
for i in data.columns:
    print(f'{i}: {data[i].isna().sum()} пропусков')

Период: 0 пропусков
Регистратор: 0 пропусков
Контрагент: 5354 пропусков
КонтрагентИНН: 208675 пропусков
СчетДт: 8 пропусков
СубконтоДт1: 2839 пропусков
ВидСубконтоДт1: 2749 пропусков
СубконтоДт2: 143393 пропусков
ВидСубконтоДт2: 141907 пропусков
СубконтоДт3: 223067 пропусков
ВидСубконтоДт3: 219437 пропусков
СчетКт: 117 пропусков
СубконтоКт1: 513 пропусков
ВидСубконтоКт1: 417 пропусков
СубконтоКт2: 18739 пропусков
ВидСубконтоКт2: 18119 пропусков
СубконтоКт3: 159364 пропусков
ВидСубконтоКт3: 153420 пропусков
ПодразделениеДт: 7907 пропусков
ПодразделениеКт: 20664 пропусков
Сумма: 106 пропусков
КоличествоДт: 334060 пропусков
КоличествоКт: 132008 пропусков
Содержание: 592 пропусков
ТипДокумента: 0 пропусков
is_manual: 0 пропусков


In [8]:
data['Контрагент'] = data['Контрагент'].fillna('Внутренняя операция')

data["КонтрагентИНН"] = data["КонтрагентИНН"].fillna("0000000000")

# Содержание — NaN допустим
data["Содержание"] = data["Содержание"].fillna("")

# Подразделения — NaN = не указано
data["ПодразделениеДт"] = data["ПодразделениеДт"].fillna("Не указано")
data["ПодразделениеКт"] = data["ПодразделениеКт"].fillna("Не указано")

# Сумма — смотрим сколько NaN осталось
print(f"Пропусков в Сумма: {data['Сумма'].isnull().sum()}")

Пропусков в Сумма: 106


In [9]:
for i in data.columns:
    print(f'{i}: {data[i].isna().sum()} пропусков')

Период: 0 пропусков
Регистратор: 0 пропусков
Контрагент: 0 пропусков
КонтрагентИНН: 0 пропусков
СчетДт: 8 пропусков
СубконтоДт1: 2839 пропусков
ВидСубконтоДт1: 2749 пропусков
СубконтоДт2: 143393 пропусков
ВидСубконтоДт2: 141907 пропусков
СубконтоДт3: 223067 пропусков
ВидСубконтоДт3: 219437 пропусков
СчетКт: 117 пропусков
СубконтоКт1: 513 пропусков
ВидСубконтоКт1: 417 пропусков
СубконтоКт2: 18739 пропусков
ВидСубконтоКт2: 18119 пропусков
СубконтоКт3: 159364 пропусков
ВидСубконтоКт3: 153420 пропусков
ПодразделениеДт: 0 пропусков
ПодразделениеКт: 0 пропусков
Сумма: 106 пропусков
КоличествоДт: 334060 пропусков
КоличествоКт: 132008 пропусков
Содержание: 0 пропусков
ТипДокумента: 0 пропусков
is_manual: 0 пропусков


In [10]:
data = data.dropna(subset=["Сумма", "СчетДт", "СчетКт"])

In [11]:
print(data.shape)

(368014, 26)


In [16]:
for i in data.columns:
    print(f'{i}: {data[i].isna().sum()} пропусков')

period: 0 пропусков
Регистратор: 0 пропусков
contractor: 0 пропусков
КонтрагентИНН: 0 пропусков
account_dt: 0 пропусков
СубконтоДт1: 2831 пропусков
ВидСубконтоДт1: 2741 пропусков
СубконтоДт2: 143385 пропусков
ВидСубконтоДт2: 141803 пропусков
СубконтоДт3: 222963 пропусков
ВидСубконтоДт3: 219323 пропусков
account_kt: 0 пропусков
СубконтоКт1: 502 пропусков
ВидСубконтоКт1: 406 пропусков
СубконтоКт2: 18728 пропусков
ВидСубконтоКт2: 18098 пропусков
СубконтоКт3: 159343 пропусков
ВидСубконтоКт3: 153303 пропусков
dept_dt: 0 пропусков
dept_kt: 0 пропусков
amount: 0 пропусков
Содержание: 0 пропусков
ТипДокумента: 0 пропусков
is_manual: 0 пропусков


In [13]:
data = data.drop(['КоличествоДт', 'КоличествоКт'], axis=1, errors='ignore')

In [14]:
data.shape

(368014, 24)

In [15]:
# data['Период'] = pd.to_datetime(data['Период'], format='%d.%m.%Y %H:%M:%S', errors='coerce')

# 2. Переименование колонок для ML
rename_dict = {
    'Период': 'period',
    'Сумма': 'amount',
    'СчетДт': 'account_dt',
    'СчетКт': 'account_kt',
    'Контрагент': 'contractor',
    'ПодразделениеДт': 'dept_dt',
    'ПодразделениеКт': 'dept_kt'
}
data = data.rename(columns=rename_dict)

# 3. Сохранение файла (обязательно!)
data.to_parquet('../data/processed/cleaned.parquet', index=False)
# data = pd.read_parquet("data/processed/cleaned.parquet")